# Create a new Pump.fun metadata dataset

Run this notebook from top to bottom to collect a new dataset. Edit only the settings cell. Everything required is written here.

The collector watches Solana mainnet for successful Pump.fun token creations. The moment the collection starts is T0. Until the enrollment period ends, every new creation is saved and its metadata is requested. The same tokens are requested again when enrollment ends, and once more after the follow-up period. A failed metadata request stays in the dataset; it does not remove the token.

Chain transactions come from `solana-rpc.publicnode.com` over IPv4. Metadata comes from the URI written in the creation transaction. If that URI contains an IPFS identifier and the first request fails, the same identifier is requested from `pump.mypinata.cloud` and then `gateway.pinata.cloud`. Local and private addresses are refused. A successful response keeps eight fields: name, symbol, description, createdOn, image, website, twitter, and telegram.


## Settings

`START_AT_UTC` is when the new collection begins. Leave it empty to start as soon as you run the next cell. To start later, enter a future UTC time ending in `Z`, such as `2026-10-03T00:00:00Z`. A past time is rejected, because this collector records tokens from the moment it starts and cannot rebuild an earlier day.

`ENROLLMENT_HOURS` is how many hours of new tokens to include. `FOLLOWUP_HOURS` is how many hours to keep observing those tokens after enrollment ends. The previous collection used 24 and 36. A shorter pair, such as 1 and 2, creates a smaller dataset and finishes sooner.

`OUTPUT_DIR` is the folder for this new dataset. Leave it empty and the notebook creates `data/live-<start time>`. If a collection stops before it finishes, paste that same folder here and run the cell again; the collector continues the unfinished dataset instead of starting a second one.


In [ ]:
# Edit this cell, then run the collection cell below.
START_AT_UTC = ""             # empty = start now; or a future time such as "2026-10-03T00:00:00Z"
ENROLLMENT_HOURS = 24         # hours of new Pump.fun creations to enroll
FOLLOWUP_HOURS = 36           # hours of observation after enrollment ends
OUTPUT_DIR = ""               # empty = create data/live-<start>; paste a folder to resume it


## Collect

The next cell prints the schedule, waits if you chose a future start, and then collects the dataset. Leave it running. It finishes only after the last checkpoint has been attempted, then prints the new dataset path and a short count of what was saved.


In [ ]:
from __future__ import annotations

import csv
import io
import json
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timedelta, timezone
from pathlib import Path

PROJECT = Path("/work/so192/scientific data")
if not (PROJECT / "code v3" / "run_mvp.py").exists():
    here = Path.cwd().resolve()
    found = [candidate for candidate in [here, *here.parents] if (candidate / "code v3" / "run_mvp.py").exists()]
    if not found:
        raise FileNotFoundError("Cannot find the code v3 folder beside this notebook.")
    PROJECT = found[0]
CODE = PROJECT / "code v3"
PROTOCOL_PATH = CODE / "configs" / "mvp_protocol.json"


def parse_utc(value: str) -> datetime:
    text = value.strip()
    if not text.endswith("Z"):
        raise ValueError("START_AT_UTC must end in Z, for example 2026-10-03T00:00:00Z.")
    return datetime.fromisoformat(text[:-1] + "+00:00").astimezone(timezone.utc)


def checkpoint_label(hours: float) -> str:
    if abs(hours - round(hours)) < 1e-9:
        return f"T+{int(round(hours))}h"
    return f"T+{int(round(hours * 60))}m"


def read_table(path: Path) -> list[dict[str, str]]:
    raw = path.read_text(encoding="utf-8", errors="replace").replace("\x00", "")
    return list(csv.DictReader(io.StringIO(raw)))


if ENROLLMENT_HOURS <= 0 or FOLLOWUP_HOURS <= 0:
    raise ValueError("ENROLLMENT_HOURS and FOLLOWUP_HOURS must both be greater than zero.")
enrollment_name = checkpoint_label(float(ENROLLMENT_HOURS))
final_name = checkpoint_label(float(ENROLLMENT_HOURS) + float(FOLLOWUP_HOURS))
if len({"T0", enrollment_name, final_name}) != 3:
    raise ValueError("Those two durations produce the same checkpoint name. Choose different lengths.")
enrollment_seconds = int(round(float(ENROLLMENT_HOURS) * 3600))
final_seconds = int(round((float(ENROLLMENT_HOURS) + float(FOLLOWUP_HOURS)) * 3600))

start_text = START_AT_UTC.strip()
now = datetime.now(timezone.utc)
if start_text:
    start_at = parse_utc(start_text)
    if start_at < now - timedelta(seconds=5):
        raise ValueError("START_AT_UTC is in the past. Leave it empty to start now, or choose a future UTC time.")
else:
    start_at = now

protocol = json.loads(PROTOCOL_PATH.read_text(encoding="utf-8"))
protocol["observation"]["checkpoints"] = ["T0", enrollment_name, final_name]
protocol["observation"]["checkpoint_offsets_seconds"] = {"T0": 0, enrollment_name: enrollment_seconds, final_name: final_seconds}
protocol["observation"]["enrollment_seconds"] = enrollment_seconds

enroll_until = start_at + timedelta(seconds=enrollment_seconds)
final_at = start_at + timedelta(seconds=final_seconds)
print("New dataset schedule")
print("  T0, collection starts: ", start_at.strftime("%Y-%m-%dT%H:%M:%SZ"))
print(" ", enrollment_name + ", enrollment ends:", enroll_until.strftime("%Y-%m-%dT%H:%M:%SZ"))
print(" ", final_name + ", last observation:", final_at.strftime("%Y-%m-%dT%H:%M:%SZ"))
print("Chain source: solana-rpc.publicnode.com")
print("Metadata: declared URI first, then pump.mypinata.cloud and gateway.pinata.cloud")

wait = (start_at - datetime.now(timezone.utc)).total_seconds()
if wait > 0:
    print(f"Waiting {wait:.0f} seconds until the chosen start.")
    time.sleep(wait)

if OUTPUT_DIR.strip():
    output = Path(OUTPUT_DIR)
    output.mkdir(parents=True, exist_ok=True)
    protocol_file = output.parent / f"{output.name}-protocol.json"
else:
    output = PROJECT / "data" / start_at.strftime("live-%Y%m%dT%H%M%SZ")
    protocol_file = output.parent / f"{output.name}-protocol.json"
output.parent.mkdir(parents=True, exist_ok=True)
protocol_file.write_text(json.dumps(protocol, indent=2) + "\n", encoding="utf-8")
print("Dataset folder:", output)
print("This cell stays busy until the last observation is attempted.")

result = subprocess.run(
    [sys.executable, "-u", "run_mvp.py", "--mode", "rpc", "--protocol", str(protocol_file), "--output", str(output)],
    cwd=CODE,
)
if result.returncode != 0:
    raise RuntimeError("Collection stopped with an error. Set OUTPUT_DIR to the dataset folder printed above and run this cell again to continue it.")

release = output / "mvp_release"
if not (release / "validation_results.json").exists():
    nested = [item for item in output.iterdir() if (item / "mvp_release" / "validation_results.json").exists()]
    if len(nested) == 1:
        release = nested[0] / "mvp_release"
        output = nested[0]
validation = json.loads((release / "validation_results.json").read_text(encoding="utf-8"))
ledger = read_table(release / "coverage_ledger.csv")
print("New dataset:", output)
print("Validation:", validation.get("status"))
print("Events:", validation.get("event_count"))
print("Observations:", validation.get("plan_rows"))
print("Final states:", validation.get("final_state_counts"))
for checkpoint, count in sorted(Counter(row["checkpoint"] + " " + row["final_state"] for row in ledger).items()):
    print(" ", checkpoint, count)


## If the cell stops early

The dataset folder printed above already contains every token and metadata attempt saved before the stop. Paste that folder into `OUTPUT_DIR` and run the collection cell again. The collector continues the same dataset from the saved clock. It does not start a second dataset and does not discard the tokens already enrolled.
